# Is the CO2 seasonal cycle getting stronger?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2FT2_getting_stronger.ipynb).

Every year the CO2 measured on Mauna Loa climbs through the northern winter and falls again
through the northern summer. That sawtooth is the northern land biosphere breathing: leaves open
in spring and pull carbon out of the air faster than everything rotting puts it back, and in
autumn the balance reverses. Almost all of the world's land is north of the equator, so the whole
atmosphere carries the signature of one hemisphere's growing season.

The swing is a few parts per million, riding on a rise of more than a hundred. The question is not
whether it is there — you will see it in the first figure — but whether it is getting **bigger**.
If northern plants are drawing down more carbon each summer than they used to, the breath should
deepen, and the record is long enough to tell.

It turns out that the answer depends on what you decide the word *amplitude* means. There are at
least three defensible answers, this notebook does not choose between them, and choosing is your
first job.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## How this notebook is different

This is a **project track**. It is not a weekly notebook and it does not behave like one.

A weekly notebook shows you a move, walks you through it, and then asks you to make it once
yourself. This one loads the data and reproduces the one thing about it that nobody disputes —
that there is a seasonal cycle, and roughly how big it is — and then stops helping. From there on
every section is a sentence describing what to find out and an empty cell to find it out in. There
is no worked example above to pattern-match against, because on a real question there never is
one.

**There is exactly one self-check in this notebook, and it is on the data loading.** After that,
nothing tells you whether you are right. That is not an oversight and it is not laziness: past the
loading step there is no single right answer here, so a cell that said `assert` would be lying to
you about how research works. What replaces it is the thing researchers actually use — a result
you can get two ways, a number you can predict before you compute it, and a claim you can try to
break.

**And it does not close.** The last section is a question this course does not know the answer to.
Everything above it is scaffolding; that question is the project.

## What you'll be able to do

**The science.** Say whether the seasonal CO2 cycle is getting stronger, with a number and an
interval rather than an adjective — and say how much of your answer came from the Earth and how
much from a definition you chose.

**The skills.** Turn a monthly record into one number per year without letting the long-term trend
contaminate it. Fit a wave to twelve points. Put an interval on a trend by resampling the years,
and use the interval to decide whether you are allowed to answer at all.

**The four questions, in order:**

1. What does the Mauna Loa record look like, and how big is its seasonal swing?
2. Is the swing bigger now than it was?
3. Does that answer survive a different definition of "amplitude"?
4. Does the same thing happen at 71° north?

The open question at the end is not on that list. It is the project; the four above are what you
build to reach it.

## Setup

NOAA's Global Monitoring Laboratory publishes one monthly-mean file per observatory, all in the
same directory and all in the same layout. Two of them are loaded below: **Mauna Loa** in Hawaii
at 19.5°N, the longest continuous CO2 record there is, and **Barrow** on the north
coast of Alaska at 71.3°N. Barrow is not analysed until the last section; it is
loaded now so that the one self-check covers both files.

**These are the *in-situ* files, and that is a choice with a cost.** The Mauna Loa record
everybody quotes — the Keeling curve, back to 1958 — is a separate NOAA *trends* file that exists
for Mauna Loa alone; this notebook reads the in-situ series instead, which begins in
1974 and so gives up the 16 years before it. What it buys is the
last section: every station NOAA runs publishes an in-situ file at this address with three letters
changed, so four stations become one measurement programme in one layout, and a rate measured here
can be compared with a rate measured there.

**Two things about these files, and the second one is a trap.**

- The measurement is the column called `value`, in parts per million, one row per month.
- A month with no measurement is not blank and it is not `NaN`. It is written as **`-999.99`**,
  which is a number, and which will happily average in with the real ones. The first thing the
  next section does is turn those into real holes — and it turns them into holes rather than
  deleting the rows, because deleting a row closes the gap and lets a twelve-month average run
  straight across it.

Mauna Loa has 12 such months and Barrow has 1. 7 of the
12 at Mauna Loa are not an instrument fault: December 2022 to June 2023 are
missing because the volcano erupted, lava crossed the access road and the observatory lost power. NOAA says so in the header
of its own trends file — *"Due to the eruption of the Mauna Loa Volcano, measurements from Mauna
Loa Observatory ... Maunakea Observatories, approximately 21 miles north"* (read 2026-08-31).
The gap in a climate record is itself Earth science.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(url, cache_name):
    """Read one station's monthly CO2 live; fall back to the copy stored with the course."""
    # Ask the live archive first. If it is down, or you are offline, read the copy stored with
    # the course instead, so the notebook still runs.
    try:
        return pd.read_csv(url, comment="#", sep=r"\s+")
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + cache_name)

GML = ("https://gml.noaa.gov/aftp/data/trace_gases/co2/in-situ/surface/txt/"
       "co2_{site}_surface-insitu_1_ccgg_MonthlyData.txt")

mauna_loa = load(GML.format(site="mlo"), "trackT2_co2_mlo.csv")
barrow = load(GML.format(site="brw"), "trackT2_co2_brw.csv")

print("Mauna Loa:", mauna_loa.shape, " Barrow:", barrow.shape)
print(mauna_loa[["year", "month", "value", "latitude"]].head())

In [ ]:
assert "value" in mauna_loa.columns and "latitude" in barrow.columns, \
    "a column the whole notebook needs is missing — the files were read wrong, or NOAA changed them"
assert len(mauna_loa) > 500 and len(barrow) > 500, \
    "expected 624-odd monthly rows at each station; far fewer means the read failed"
print(f"✓ the data — {len(mauna_loa)} monthly rows at Mauna Loa "
      f"({mauna_loa.latitude.iloc[0]}°N) and {len(barrow)} at Barrow "
      f"({barrow.latitude.iloc[0]}°N), of which {(mauna_loa.value < 0).sum()} and "
      f"{(barrow.value < 0).sum()} are marked -999.99 for 'no measurement'")

### And that is the last self-check in this notebook

The pipeline is now trustworthy: the files are the files, the columns are the columns, the numbers
below are the numbers. Everything from here is yours, and nothing will tell you when you have it
right.

## What does the Mauna Loa record look like, and how big is its seasonal swing?

Two things are happening in this record at once and they have to be separated before anything can
be measured. There is a **trend** — CO2 today is far above CO2 in the 1970s — and there is a
**swing** around it that repeats every year.

The trend is easy to estimate without any curve fitting at all. Average twelve consecutive months
and the seasons cancel, because each one appears exactly once; slide that average along the record
and what comes out is the trend with the seasons taken out of it. Subtract it and what is left is
the swing.

Two kinds of hole appear on the way. The -999.99 months become holes, because that is what they
are; and the subtraction leaves one at each end, since the first six months and the last six have
no full year around them to average.
**NaN:** Where the file had nothing at all, pandas puts NaN. A NaN is a hole, not a zero.

In [ ]:
def seasonal(station):
    """Turn the months with no measurement into real holes, then split CO2 into trend and swing."""
    station = station.copy()
    station.loc[station["value"] < 0, "value"] = np.nan
    station["trend"] = station["value"].rolling(12, center=True).mean()
    station["swing"] = station["value"] - station["trend"]
    return station


mauna_loa = seasonal(mauna_loa)
barrow = seasonal(barrow)

print("Mauna Loa, months in the file:", len(mauna_loa),
      "from", mauna_loa.year.min(), "to", mauna_loa.year.max())
print("of those, months with a real measurement:", mauna_loa.value.notna().sum())
print("and months with a trend to subtract:", mauna_loa.swing.notna().sum())

The whole record, with the twelve-month average drawn through it. The sawtooth is the thing this
project is about; the smooth line is what has to come off before it can be measured.

In [ ]:
plt.plot(mauna_loa["year"] + mauna_loa["month"] / 12, mauna_loa["value"],
         color="0.4", lw=0.8)
plt.plot(mauna_loa["year"] + mauna_loa["month"] / 12, mauna_loa["trend"],
         color="firebrick", lw=1.4)
plt.xlabel("year")
plt.ylabel("CO$_2$ (ppm)")
plt.title(f"Mauna Loa monthly CO$_2$ and its 12-month average "
          f"(n = {len(mauna_loa)} months)")
plt.show()

Now the swing on its own, averaged over every year in the record — one number per calendar month.
This is the shape the northern growing season leaves in the atmosphere.

In [ ]:
profile = mauna_loa.groupby("month")["swing"].mean()

plt.bar(profile.index, profile.values, color="0.4")
plt.axhline(0, color="firebrick", lw=1.2)
plt.xlabel("month (1 = January)")
plt.ylabel("average swing about the trend (ppm)")
plt.title(f"The average seasonal swing at Mauna Loa ({mauna_loa.year.min()}"
          f"-{mauna_loa.year.max()})")
plt.locator_params(axis="x", integer=True)
plt.show()

print("highest month:", profile.idxmax(), round(profile.max(), 2), "ppm")
print("the two lowest months:", profile.sort_values().head(2).round(2).to_dict())
print("top to bottom:", round(profile.max() - profile.min(), 2), "ppm")

That is the Keeling seasonal cycle, and it is the one result this notebook hands you: a peak in
**May** at +3.19 ppm, a trough shared between **September**
and **October** — -3.26 and -3.26, which fifty
years of this record cannot separate — and about **6.5 ppm** between top and
bottom. Northern plants draw carbon down through the summer and the atmosphere reaches its lowest
point just as the growing season ends.

Nothing above is in dispute. Everything below is.

## Is the swing bigger now than it was?

One average cycle over the whole record cannot answer that. You need **one number per year**, and
then a line through those numbers.

Two decisions are forced on you before you can write a single line, and both matter more than they
look. The first is what "one number" means for a year — that is the next section's whole subject,
so for now take the most obvious answer you can think of. The second is which years you are
allowed to use at all: a year missing four months has a smaller range than a year with twelve, for
no reason to do with plants, so an incomplete year is not a smaller amplitude but a missing one.

### ✏️ Your turn 1

Write a function that turns a station's table into one row per year — the year, and how far its
CO2 swings between the highest month and the lowest — and use it on Mauna Loa.

**Use these names**, because every later section reuses them: the function `amplitude(station,
column)`, taking the table and the name of the column to measure, and returning a table with the
two columns `year` and `amplitude`. Every definition you try later has to hand back that same
shape, so that one piece of fitting code works on all of them.

Use only years with all twelve months. Then plot amplitude against year, fit a straight line to
it, and print the slope in **ppm per decade** (`.coef_[0]` is per year).

**Linear regression:** Draw the best straight line. Best means the smallest total miss.

Then print one more line answering it in a sentence, on your own slope: is the seasonal swing at
Mauna Loa getting stronger?

In [ ]:
# ← your answer here



A slope on its own is not an answer yet. Whatever came out of your fit, it is a small fraction of
the mean amplitude you printed beside it, and the cloud of points around the line is wide. A
single fitted slope with no interval on it cannot tell you whether that is a small effect or no
effect.

### ✏️ Your turn 2

Put an interval on that slope by resampling the years.

**Bootstrap:** Ask the data the same question a thousand times, using a different random slice of itself each time.

The recipe, in words. 2000 times over: draw 48-odd years from your
amplitude table **with replacement** — `amps.sample(n=len(amps), replace=True, random_state=i)` —
refit the line to that resample, and collect the slope. The `random_state` is what makes your
interval the same every time you run the cell; without it the number in your write-up and the
number in your notebook will not match. Write it as a function `trend_spread(amps)` that hands
back the whole array of slopes, and while you are there write `trend(amps)` for the slope alone,
because you will call both on four more series before the notebook is over.

Report the 2.5th and 97.5th percentiles, and the fraction of the resamples that came out above
zero. Draw the 2000 slopes as a histogram with zero marked.

**Confidence interval:** Not one number but the range your number would have wandered over, had the world rolled differently.

Then print two or three sentences answering it on your own interval: are you able to say whether
the Mauna Loa seasonal cycle is getting stronger, and what would your interval have to look like
before you could?

In [ ]:
# ← your answer here



So the obvious definition gives no answer. The interval you printed straddles zero and the share
of resamples above it is not far off a coin toss, so on the complete years of the cleanest CO2
record there is, the question in the title has just come back *don't know*.

That is a real result and you could stop there. It would also be wrong.

### Predict before you run

You are about to measure the same thing two other ways. Both are defensible, and neither is more
obviously correct than what you have just done.

How far could the answer move? Write down the slope you think the *most different* of the three
definitions will give, in ppm per decade — the answer you already have is in the cell above to
measure it against. Change `my_guess` and run the cell. You will check it at the end of the next
section, and a wrong guess you committed to is worth more than a right answer you were shown.

In [ ]:
my_guess = None    # ← your number, written down before you look

In [ ]:
assert my_guess is not None, \
    "write a number into my_guess before you run this — the commitment is the point, "\
    "and a guess you made before you saw the answer is the only one that can teach you "\
    "anything"
print("✓ committed — I think the most different definition will give", my_guess, "ppm per decade")

## Does that answer survive a different definition of "amplitude"?

You measured the swing of the **raw** record: the highest month of a calendar year minus the
lowest. That is one answer to "how big is the cycle this year", and there are at least two others.

- **The raw range.** Highest month minus lowest, straight off `value`. What you already did.
- **The detrended range.** The same, but of `swing` — the record with the twelve-month average
  already subtracted.
- **The size of the once-a-year wave.** Fit a smooth wave with exactly one cycle per year to each
  year's twelve swing values and measure the fitted wave instead of the data. A wave through
  twelve points is a straight-line fit like any other, with two columns instead of one:
  `np.sin(2 * np.pi * month / 12)` and `np.cos` of the same angle. The fitted values come back
  from `.predict`, and the amplitude is their range.

This is the one real decision in this track. Make it, and report what it cost.

### ✏️ Your turn 3

Measure the Mauna Loa amplitude all three ways.

The third one needs a column that does not exist yet. Write `fourier_fit(station)` that adds a
`fourier` column holding, for every month, the height of that year's best-fitting once-a-year
wave — loop over `station.groupby("year")`, fit `LinearRegression` to the two wave columns against
`swing`, and write `.predict` back into the rows you fitted. Give it the same shape as `seasonal`:
a station table in, a station table out. Then all three definitions are one call each,
`amplitude(station, column)`, and your `trend` and `trend_spread` work on all three unchanged.

Put the three amplitude series on one plot. Then print, for each: the trend in ppm per decade, its
95% interval, and the fraction of resamples above zero.

Compare the spread of the three answers with the number you committed to in *Predict before you
run*. Then print one more line answering it in a sentence: does the answer to this notebook's
title depend on which definition you chose, and which one would you report?

In [ ]:
# ← your answer here



Three defensible definitions, and they do not merely differ in size — **they differ in what the
answer is.** Two of the three put their whole interval on one side of zero and one does not, and
the one that does not is the definition you would have written first.

Two definitions agreeing and one disagreeing is a stronger clue than three-way disagreement would
be: it says something specific is wrong with the odd one out.

### ✏️ Your turn 4

Find out what. The raw range is smaller than the detrended range in every single year — go and
check that it is — so the trend must be eating part of the swing. Work out how much, and see
whether that accounts for the whole disagreement.

Here is the mechanism to test, in words. Within one calendar year the peak comes first and the
trough comes months later. Between them, the long-term rise has lifted the whole record a little,
so the trough is measured higher than it would have been and the raw range comes out short. The
size of that theft is roughly the year's own rise multiplied by the fraction of a year between
peak and trough: **`lag / 12 × growth`**, where `lag` is the number of months from the highest
month to the lowest and `growth` is how much CO2 rose that year.

Build all three per year — the gap between the two amplitudes, the lag, and the growth — and put
the predicted theft beside the observed gap. `idxmax` and `idxmin` on a group give you the row of
the highest and lowest month; the year's growth is the difference between neighbouring annual
means, halved.

One trap in that last step. Some years are not complete and drop out, so "the year before" and
"the year after" are not the row above and the row below. Reindex your annual means over the whole
range of years first — `.reindex(range(first, last + 1))` — or the eruption gap will silently make
2021 and 2024 neighbours and report three years of rise as two.

Then print one more line answering it in a sentence, on your own numbers: does the theft account
for the gap between the two definitions, and does it account for the difference between their two
*trends* as well?

In [ ]:
# ← your answer here



The gap you measured is larger in the recent years than in the early ones, and the one-line
prediction tracks it — window against window, and year against year in the correlation you
printed. The trough at Mauna Loa arrives several months after the peak, and CO2 rises faster now
than it did in the 1970s, so the raw definition loses more of the swing every decade than it used
to. Put your rate for the gap itself beside the difference between the two definitions' trends:
they are the same number to the precision either of them deserves.

**The raw definition did not measure a cycle that is not growing. It measured a cycle that is
growing, minus a theft that is growing by about the same amount.** That is not a coincidence you
could have guessed; it is a fact about this record that had to be computed.

## Does the same thing happen at 71° north?

Mauna Loa is at 19.5°N, in the middle of an ocean and 3.4 km
up — its `elevation` column says so. Almost none
of the land whose plants make the seasonal cycle is anywhere near it; what it measures is air that
has been stirred across a hemisphere.

Barrow sits on the Arctic coast of Alaska at 71.3°N, among the tundra and boreal
forest that do the breathing. It has been loaded since the setup cell and has had `seasonal`
applied to it. Everything you have written works on it unchanged.

### ✏️ Your turn 5

Run the whole of the last three sections again at Barrow, and change nothing but the station.

Add the `fourier` column, take all three amplitude series, fit and bootstrap each, and print the
same numbers you printed for Mauna Loa. Put the Barrow and Mauna Loa amplitude series on one
plot — one definition is enough for the figure, as long as you say which.

Also print, for both stations, the percentage change from the first fifteen complete years to the
last fifteen, so you have both ways of stating the same result on the page.

Then print one more line answering it in a sentence: does the choice of definition matter as much
at Barrow as it did at Mauna Loa, and what does the pair of stations together suggest?

In [ ]:
# ← your answer here



At Barrow the fork closes. Your three definitions land within a fraction of a ppm per decade of
each other, every interval sits clear of zero, and the Barrow rate is a large multiple of the
Mauna Loa one on a cycle that was already several times as deep.

The definition mattered at Mauna Loa and does not matter here, and the reason is arithmetic rather
than geography: the theft is roughly the same size at both stations, and at Barrow it is a few
percent of a large number instead of all of a small one. **A choice that changes your conclusion at
one station and nothing at another is not a detail of method. It is a measure of how thin your
signal was.**

### ✏️ Your turn 6

Two or three paragraphs, quoting **your own numbers from Your turn 5** — both the ppm per decade
and the percentage change.

1. You now have two ways to say the same result: a rate in ppm per decade, and a percentage change
   from your first fifteen years to your last fifteen. Which of the two can be compared honestly
   between two stations, and which cannot? Work out what happens to the percentage if you move the
   two windows closer together, and say what that means for a sentence of the form *"Mauna Loa
   gives +13% and Barrow gives +25%"*.
2. Your Barrow interval excludes zero by a wide margin and your Mauna Loa raw interval does not.
   Name what a reader should conclude from a result that changes side when you change a defensible
   choice, and say what would have to be true of the Mauna Loa record for the three definitions to
   agree there the way they do at Barrow.

*(Double-click this cell and replace this line with your answer.)*

## The question, answered

At Barrow, unambiguously yes: every definition agrees there, and every interval you computed sits
clear of zero. At Mauna Loa yes, but only once the rising trend is taken out first — your
detrended interval excludes zero and your raw one, fitted on the same record, cannot tell growth
from nothing. Which of those two sentences you are entitled to write was decided by a definition
you chose.

## What track T2 leans on

**The question.** Is the CO2 seasonal cycle getting stronger?

Nothing here is new. These are the weeks to look back at while you work, and the wording is the course's own.

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Linear regression** | Draw the best straight line. Best means the smallest total miss. |
| **Bootstrap** | Ask the data the same question a thousand times, using a different random slice of itself each time. |
| **Confidence interval** | Not one number but the range your number would have wandered over, had the world rolled differently. |
| **NaN** | Where the file had nothing at all, pandas puts NaN. A NaN is a hole, not a zero. |
| **Table** | A table with a name on every column, so you ask for data by name instead of by position. |

### Code you will reach back for

| Function | What it does |
|---|---|
| `table.groupby(column)` | split the table into one group per value |
| `table.dropna()` | throw away every row with a hole anywhere in it |
| `series.rolling(365).sum()` | slide a window of that many rows along and total what is inside it |
| `series.idxmax()` | the LABEL of the largest value — here, the date of the busiest day |
| `column.mean()` | the average of a column |
| `table.tail(n)` | the last n rows |
| `LinearRegression().fit(x, y)` | find the straight line with the smallest total miss |
| `model.coef_[0]` | the slope of the fitted line |
| `model.predict(x)` | what the fitted line says y should be at each x |
| `table.sample(n, replace=True)` | draw a new table the same size, picking rows at random and putting each back |
| `np.percentile(values, [2.5, 97.5])` | the two values that cut off the bottom and top 2.5% — a 95% interval |

## What your project must contain

Five sections, empty below, required of **every** EPS 88 project regardless of track. They are
headed here so the shape of a good answer is visible while you work. Fill them in as you go; they
are not a write-up you do at the end.

### ✏️ 1 · A one-sentence answer

Your claim and its uncertainty, in one sentence, at the top of your report. If you cannot put a
number and a range in it, you do not have a result yet. On this track the range is not optional
decoration — one of the three definitions gave an interval containing zero, and a sentence without
an interval could not have said so.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 2 · The trivial baseline

Before any statistic, state the dumbest answer to your question and what it gives. Every later
number is reported against it.

On this track the baseline is the raw peak-to-trough range: no detrending, no fitting, the
definition anybody would write first. Say what it gives, and say exactly what each later step
bought you over it — and where it bought you nothing.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 3 · Split by structure

Earth data are correlated in space and in time, so whatever you split, resample or count as
independent has to be split along the structure that is really there — never at random across
rows.

This track fits no model, so there is no train/test split to get wrong. The same idea has teeth
anyway: every interval you quoted came from resampling **years**, which assumes one year's
amplitude tells you nothing about the next one's. Name the unit you treated as independent, say
why, and say what you would have to do differently if neighbouring years turned out to move
together.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 4 · What I got wrong

What failed, and what you believed before it failed. Honest failure is graded; a faked success is
not. Your *Predict before you run* guess belongs here if it was wrong.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 5 · AI disclosure

Which tool, what you asked it, what you changed in what it gave you, and how you checked that the
result was true.

*(Double-click this cell and replace this line with your answer.)*

## The open question

> **How far north does it scale, and what does it say about which ecosystems are doing the breathing?**

Nobody grading this knows the answer, and neither does the literature. Everything above is the
scaffolding; this is the project.

Here is what is actually established, and it is less than it looks. Two stations do not make a
gradient. You have one detrended rate at 19.5°N and one at 71.3°N, and
a straight line through two points fits perfectly and means nothing. They are not even quite a
pair: each was fitted over its own station's complete years, so setting them side by side means
first deciding which years both are allowed to use — and that decision is not free. What is
**not** settled is the shape between them and beyond them — whether the deepening rises smoothly
with latitude, switches on somewhere, or tracks something else entirely that happens to correlate
with latitude.

Three directions, none of them worked out here:

1. **Add stations.** Every observatory NOAA runs publishes a file at the same address with three
   letters changed, in the same layout, so `load` and every function you wrote work on all of them.
   The in-situ set is small — Samoa (`smo`), the South Pole (`spo`), Maunakea (`mko`) — but there
   is a much larger flask-sampling set at
   `.../co2/flask/surface/txt/co2_SITE_surface-flask_1_ccgg_month.txt`, including Alert at 82°N and
   dozens between. Those files carry no header row, so `pd.read_csv` needs
   `names=["site", "year", "month", "value"]`; that is the only change. How many stations would
   you need before the shape of the curve, rather than its two endpoints, were established?
2. **Ask what the gradient is a gradient in.** Latitude is a proxy for several things at once —
   how much land there is, how long the growing season is, how far the air has travelled from where
   the carbon was taken up. The southern stations are the test: the South Pole is as far from the
   equator as it is possible to be and has almost no land anywhere near it. If the deepening
   follows latitude it should be large there; if it follows land it should be tiny. Both are
   computable with what you have already written.
3. **Separate a deeper breath from a longer one.** A cycle can grow because the summer drawdown is
   stronger or because the growing season is longer, and those are different claims about
   ecosystems. The month of the trough is in your data; so is the width of the drawdown. Does the
   trough arrive later than it used to, and does that vary with latitude?

And one that is bigger than a semester: **which ecosystems are actually doing the breathing?** The
atmosphere at any station is a mixture stirred from everywhere upwind, so a station's amplitude is
not a measurement of the vegetation beneath it. What would you need — more stations, or a way of
saying where the air had been — before a CO2 record could name a biome rather than a latitude? If
the answer is that no arrangement of surface stations can do it, that is a result, and it is worth
saying carefully.

### ✏️ Your turn 7 — the first move

Before you close this notebook: in a few sentences, name the **one** measurement you would make
first, say what it would show if the deepening really does scale with latitude, what it would show
if it does not, and name the number that would change your mind. Then make it, in the cell below
the prose.

*(Double-click this cell and replace this line with your answer.)*

In [ ]:
# ← your answer here

